# 09 — Equivariance Reliance: SO(3) Rotation & Translation Robustness

From SUMMER_PLAN.md's "Full-Body Rotation and Translation Test": since the
models are not built with true SE(3) equivariance, this notebook tests
empirically whether prediction quality is stable under random rotations and
translations of the input geometry. If stable, that's a strong argument
against the complexity cost of a Tier-5 equivariant architecture; if not,
it motivates coordinate augmentation during training and/or that investment.

**This is an evaluation-only notebook — no training happens here.** It
reuses the four checkpoints from notebook 08 (Phase 2: `{distance,
attention} x {features on, features off}`, each using its architecture's
Phase 1 winning aggregation) and evaluates them under transformed inputs.
Positioned after notebook 08 rather than at the end of the study because a
failure here would call the entire non-equivariant architecture into
question — worth knowing before sinking more compute into Phases 3+.

## Why this might already be a non-issue for half the models

Every `MessageLayer` edge feature is an RBF encoding of a pairwise
*distance*, and `AtomEncoder` uses only categorical atom/residue/bond-count
features — none of these change under a rigid rotation+translation of the
whole structure. The **one** exception is `QueryEncoder`'s optional
`normal` feature: it's fed as raw x/y/z components into an MLP with no
equivariant handling, so it *does* change under rotation.

That means the `features off` checkpoints (no `normal`, no `curvature`)
should be **exactly** invariant by construction — not an empirical
discovery, a mathematical consequence of the input feature set. The
`features on` checkpoints are the genuinely open question. Section 2 below
verifies this reasoning numerically on a real cached graph before running
the full sweep, so the rest of the notebook isn't just trusting the
argument.

## Method

For each checkpoint, each of several transform types, and multiple random
draws per type, per test protein:
1. Apply the transform to the cached graph's `atom.pos` / `query.pos` (and
   rotate `query.normal` if present — directions rotate but don't
   translate).
2. Recompute the `radial`/`aq`/`qq` edges (kNN + RBF) from the transformed
   positions using the same functions `graph_builder.py` uses at graph-build
   time — not just re-using the cached edges, so this is a genuine
   re-derivation of the graph representation, not an assumption that it's
   unchanged. Bond edges are left untouched (bond distances are invariant;
   recomputing them would just reproduce the same values).
3. Run the frozen model on both the original and transformed graph, compare
   predictions directly — this is the actual invariance metric, independent
   of ground-truth accuracy.

**Transform types:** `translation` (negative control — nothing in the
feature set is position-absolute, so this should show exactly zero effect
regardless of model; a failure here would indicate a bug, not a real
finding), `rotation` (the real test), `rotation+translation` (realistic
worst case, both applied together).

## Prerequisites

- [ ] Phase 1 (notebook 07) and Phase 2 (notebook 08) concluded, winning
      `agg` and features on/off checkpoints exist for both architectures


## 1. Setup: imports and rigid-transform helpers

In [ ]:
import copy
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.spatial.transform import Rotation

sys.path.insert(0, str(Path("../..").resolve()))

from src.analysis.embedding_analysis import load_model_frozen, _load_graph
from src.data.dataset import load_split_manifest
from src.data.graph_builder import _knn_radial, _knn_bipartite, _knn_self, _rbf_encode
from src.utils.config import get_data_root
from src.utils.paths import ProteinPaths

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = get_data_root()
print(f"DEVICE: {DEVICE}")

In [ ]:
def _bond_set_from_edge_index(edge_index: torch.Tensor) -> set[tuple[int, int]]:
    """Reconstruct the undirected (min,max) bond-pair set the radial kNN
    exclusion needs, from the cached graph's directed bond edge_index."""
    src = edge_index[0].tolist()
    dst = edge_index[1].tolist()
    return {(min(i, j), max(i, j)) for i, j in zip(src, dst)}


def apply_rigid_transform(data, R: np.ndarray, t: np.ndarray,
                           knn_radial: int = 16, knn_aq: int = 32, knn_qq: int = 8):
    """
    Return a NEW HeteroData with atom/query positions rigidly transformed
    (x -> x @ R.T + t) and the radial/aq/qq edges genuinely recomputed from
    the transformed geometry (same kNN + RBF functions graph_builder.py uses
    at build time) — not just reusing the cached edges.

    Bond edges (topology + edge_attr) are left untouched: bond distances are
    invariant to rigid transforms, so recomputing them would reproduce the
    same values. curvature (scalar, invariant) is also left untouched.
    normal (if present) is rotated but not translated, since it's a
    direction, not a position.

    `data` must be CPU-resident (raises via .numpy() otherwise — don't pass
    a graph that's already been moved to a device, e.g. by predict_esp).
    """
    new_data = copy.deepcopy(data)

    atom_xyz  = data["atom"].pos.numpy()
    query_xyz = data["query"].pos.numpy()

    new_atom_xyz  = (atom_xyz  @ R.T) + t
    new_query_xyz = (query_xyz @ R.T) + t

    new_data["atom"].pos  = torch.tensor(new_atom_xyz,  dtype=torch.float)
    new_data["query"].pos = torch.tensor(new_query_xyz, dtype=torch.float)

    if hasattr(data["query"], "normal"):
        normal     = data["query"].normal.numpy()
        new_normal = normal @ R.T
        new_normal = new_normal / np.linalg.norm(new_normal, axis=-1, keepdims=True)
        new_data["query"].normal = torch.tensor(new_normal, dtype=torch.float)

    n_rbf = data["atom", "radial", "atom"].edge_attr.shape[1]
    bond_set = _bond_set_from_edge_index(data["atom", "bond", "atom"].edge_index)

    radial_src, radial_dst, radial_dists = _knn_radial(new_atom_xyz, knn_radial, bond_set)
    aq_src, aq_dst, aq_dists             = _knn_bipartite(new_atom_xyz, new_query_xyz, knn_aq)
    qq_src, qq_dst, qq_dists             = _knn_self(new_query_xyz, knn_qq)

    radial_rbf = _rbf_encode(radial_dists, n_rbf, d_min=1.8, d_max=8.0)
    aq_rbf     = _rbf_encode(aq_dists,     n_rbf, d_min=0.0, d_max=12.0)
    qq_rbf     = _rbf_encode(qq_dists,     n_rbf, d_min=0.0, d_max=8.0)

    new_data["atom", "radial", "atom"].edge_index = torch.tensor(
        np.stack([radial_src, radial_dst]), dtype=torch.long)
    new_data["atom", "radial", "atom"].edge_attr = torch.tensor(radial_rbf, dtype=torch.float)

    new_data["atom", "aq", "query"].edge_index = torch.tensor(
        np.stack([aq_src, aq_dst]), dtype=torch.long)
    new_data["atom", "aq", "query"].edge_attr = torch.tensor(aq_rbf, dtype=torch.float)

    new_data["query", "qq", "query"].edge_index = torch.tensor(
        np.stack([qq_src, qq_dst]), dtype=torch.long)
    new_data["query", "qq", "query"].edge_attr = torch.tensor(qq_rbf, dtype=torch.float)

    return new_data


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniform random SO(3) rotation matrix."""
    return Rotation.random(random_state=rng).as_matrix()


def random_translation(rng: np.random.Generator, scale: float = 50.0) -> np.ndarray:
    """Random translation vector, uniform in [-scale, scale] per axis (Å)."""
    return rng.uniform(-scale, scale, size=3)


def get_transform(transform_type: str, rng: np.random.Generator):
    """Return (R, t) for a named transform type. 'none' -> identity."""
    I3 = np.eye(3)
    zero = np.zeros(3)
    if transform_type == "none":
        return I3, zero
    if transform_type == "translation":
        return I3, random_translation(rng)
    if transform_type == "rotation":
        return random_rotation(rng), zero
    if transform_type == "rotation+translation":
        return random_rotation(rng), random_translation(rng)
    raise ValueError(f"unknown transform_type: {transform_type}")


@torch.no_grad()
def predict_esp(model, data, esp_mean: float, esp_std: float, device) -> np.ndarray:
    """Forward pass + denormalize to raw kT/e (matches Trainer.evaluate_test).

    Clones before moving to device — HeteroData.to(device) mutates in place,
    so without the clone, the caller's CPU graph would end up device-resident
    and break any later apply_rigid_transform(data, ...) call on the same
    reference (its .numpy() calls require CPU tensors).
    """
    data_on_device = data.clone().to(device)
    pred_norm = model(data_on_device)
    pred_raw  = pred_norm * esp_std + esp_mean
    return pred_raw.cpu().numpy()


print("Helpers defined.")

## 2. Sanity check: verify the invariance argument numerically

Before running the full sweep, confirm on one real protein that (a) the
radial/aq/qq RBF edge features really are numerically invariant under a
random rigid transform, (b) bond edges are untouched, and (c) normal
vectors (when present) genuinely do change. If any of this doesn't hold,
the rest of the notebook's results aren't trustworthy.

In [ ]:
_, _, _sanity_test_ids = load_split_manifest(DATA_ROOT)
SANITY_PID = _sanity_test_ids[0]

_sanity_data = _load_graph(SANITY_PID, DATA_ROOT)
print(f"Sanity-check protein: {SANITY_PID}")
print(f"  atom nodes: {_sanity_data['atom'].pos.shape[0]}   query nodes: {_sanity_data['query'].pos.shape[0]}")
print(f"  has normal: {hasattr(_sanity_data['query'], 'normal')}")
print(f"  has curvature: {hasattr(_sanity_data['query'], 'curvature')}")

_rng = np.random.default_rng(42)
_R, _t = get_transform("rotation+translation", _rng)
_transformed = apply_rigid_transform(_sanity_data, _R, _t)

print("\n--- Edge count check (should match exactly) ---")
for et in [("atom", "radial", "atom"), ("atom", "aq", "query"), ("query", "qq", "query")]:
    n_orig = _sanity_data[et].edge_index.shape[1]
    n_new  = _transformed[et].edge_index.shape[1]
    print(f"  {et}: orig={n_orig}  transformed={n_new}  match={n_orig == n_new}")

print("\n--- RBF invariance check (radial edges — should be ~0 diff) ---")
_orig_rbf = _sanity_data["atom", "radial", "atom"].edge_attr.numpy()
_new_rbf  = _transformed["atom", "radial", "atom"].edge_attr.numpy()
if _orig_rbf.shape == _new_rbf.shape:
    _diff = np.abs(_orig_rbf - _new_rbf)
    print(f"  max abs diff: {_diff.max():.2e}   mean abs diff: {_diff.mean():.2e}")
else:
    print(f"  SHAPE MISMATCH — kNN topology changed under rotation (rare tie-break flip)")

print("\n--- Bond edge invariance check (should be identical) ---")
_bond_same = torch.equal(_sanity_data["atom", "bond", "atom"].edge_attr,
                          _transformed["atom", "bond", "atom"].edge_attr)
print(f"  bond edge_attr identical: {_bond_same}")

if hasattr(_sanity_data["query"], "normal"):
    print("\n--- Normal vector check (should differ — normals rotate) ---")
    _n_orig = _sanity_data["query"].normal.numpy()
    _n_new  = _transformed["query"].normal.numpy()
    print(f"  max abs diff: {np.abs(_n_orig - _n_new).max():.4f}")
else:
    print("\n(no normal feature on this cached graph)")

## 3. Configuration

`RUNS` mirrors notebook 08's Phase 2 checkpoints exactly — same 4 models,
no retraining. `PROTEIN_SAMPLE_SIZE` caps how many test-set proteins are
swept per condition (set to `None` to use the full test set — the kNN
recompute + forward pass are both cheap, so this is mainly a knob for a
quick first pass rather than a hard compute limit).

In [ ]:
THESIS_ROOT = Path("/home/student/thesis")
CKPT_ROOT   = THESIS_ROOT / "checkpoints"

RUNS = [
    dict(label=f"{model.capitalize()} — features {onoff}", model_type=model, features=onoff,
         ckpt_dir=CKPT_ROOT/f"{model}_features_{onoff}")
    for model in ["attention", "distance"]
    for onoff in ["off", "on"]
]

TRANSFORM_TYPES     = ["translation", "rotation", "rotation+translation"]
N_REPEATS_PER_TYPE  = 5
PROTEIN_SAMPLE_SIZE = 50    # None = full test set
SEED                = 123

_, _, TEST_IDS = load_split_manifest(DATA_ROOT)
if PROTEIN_SAMPLE_SIZE is not None:
    _rng_sample = np.random.default_rng(SEED)
    TEST_IDS = list(_rng_sample.choice(TEST_IDS, size=min(PROTEIN_SAMPLE_SIZE, len(TEST_IDS)), replace=False))

print(f"{'Run':<28}  {'Checkpoint exists':>18}")
print("-" * 48)
for r in RUNS:
    print(f"{r['label']:<28}  {'yes' if r['ckpt_dir'].exists() else 'no':>18}")
print(f"\nTest proteins in sweep: {len(TEST_IDS)}")
print(f"Transform types: {TRANSFORM_TYPES}  x  {N_REPEATS_PER_TYPE} repeats each")

## 4. Full sweep

For every checkpoint, every protein, every transform type, and every repeat:
build the transformed graph, run the frozen model on both original and
transformed inputs, and record (a) how much the *prediction itself* shifted
— the direct invariance metric — and (b) how accuracy (Pearson r / RMSE vs
ground truth) compares before and after.

In [ ]:
def pearsonr_np(a, b):
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


rows = []

for run in RUNS:
    if not run["ckpt_dir"].exists():
        print(f"Skipping {run['label']} — checkpoint not found at {run['ckpt_dir']}")
        continue

    model, ckpt = load_model_frozen(run["ckpt_dir"], DEVICE)
    esp_mean, esp_std = ckpt["esp_mean"], ckpt["esp_std"]
    print(f"Loaded {run['label']}  (esp_mean={esp_mean:.4f}  esp_std={esp_std:.4f})")

    for pid in TEST_IDS:
        try:
            data = _load_graph(pid, DATA_ROOT)
        except FileNotFoundError:
            continue
        true_esp = data["query"].y.numpy()

        pred_orig = predict_esp(model, data, esp_mean, esp_std, DEVICE)
        orig_r    = pearsonr_np(pred_orig, true_esp)
        orig_rmse = float(np.sqrt(np.mean((pred_orig - true_esp) ** 2)))

        rng = np.random.default_rng(hash((pid, run["label"])) % (2**32))

        for transform_type in TRANSFORM_TYPES:
            for rep in range(N_REPEATS_PER_TYPE):
                R, t = get_transform(transform_type, rng)
                transformed = apply_rigid_transform(data, R, t)
                pred_trans  = predict_esp(model, transformed, esp_mean, esp_std, DEVICE)

                trans_r    = pearsonr_np(pred_trans, true_esp)
                trans_rmse = float(np.sqrt(np.mean((pred_trans - true_esp) ** 2)))

                rows.append({
                    "Run":            run["label"],
                    "Model":          run["model_type"].capitalize(),
                    "Features":       run["features"],
                    "Transform":      transform_type,
                    "Protein":        pid,
                    "Repeat":         rep,
                    "PredPred corr":  pearsonr_np(pred_orig, pred_trans),
                    "PredPred RMSE":  float(np.sqrt(np.mean((pred_orig - pred_trans) ** 2))),
                    "Orig r":         orig_r,
                    "Transformed r":  trans_r,
                    "Orig RMSE":      orig_rmse,
                    "Transformed RMSE": trans_rmse,
                })

results_df = pd.DataFrame(rows)
print(f"\nCollected {len(results_df)} (checkpoint x protein x transform x repeat) rows.")

## 5. Aggregate: does prediction stability differ by transform type or model?

**`PredPred corr`** (correlation between the same protein's original-input
and transformed-input predictions) is the direct invariance metric — 1.0
means the transform had zero effect. This is what Section 2's construction
argument predicts should be exactly 1.0 for `translation` on every model,
and for `rotation`/`rotation+translation` on the `features off` models.

In [ ]:
if results_df.empty:
    print("No results — checkpoints not found. Run notebook 08's sweep first.")
else:
    summary = (
        results_df
        .groupby(["Run", "Transform"])
        .agg(
            mean_predpred_corr=("PredPred corr", "mean"),
            std_predpred_corr=("PredPred corr", "std"),
            mean_predpred_rmse=("PredPred RMSE", "mean"),
            mean_orig_r=("Orig r", "mean"),
            mean_transformed_r=("Transformed r", "mean"),
        )
        .round(6)
    )
    pd.set_option("display.max_rows", 100)
    display(summary)

In [ ]:
if not results_df.empty:
    runs_present  = sorted(results_df["Run"].unique())
    transforms    = TRANSFORM_TYPES

    fig, axes = plt.subplots(1, len(runs_present), figsize=(5 * len(runs_present), 5), sharey=True)
    if len(runs_present) == 1:
        axes = [axes]
    fig.suptitle("Prediction stability under rigid transforms\n"
                 "(PredPred corr — 1.0 = transform had zero effect on predictions)",
                 fontsize=12, fontweight="bold")

    for ax, run_label in zip(axes, runs_present):
        sub = results_df[results_df["Run"] == run_label]
        data_by_transform = [sub[sub["Transform"] == t]["PredPred corr"].dropna().values for t in transforms]
        ax.boxplot(data_by_transform, tick_labels=transforms)
        ax.set_title(run_label, fontsize=10)
        ax.set_ylabel("corr(pred_orig, pred_transformed)")
        ax.tick_params(axis="x", rotation=20)
        ax.grid(axis="y", alpha=0.3)
        ax.axhline(1.0, color="green", lw=1, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.show()

In [ ]:
if not results_df.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    plot_summary = (
        results_df.groupby(["Run", "Transform"])["Orig r"].mean().reset_index()
        .merge(results_df.groupby(["Run", "Transform"])["Transformed r"].mean().reset_index(),
               on=["Run", "Transform"])
    )
    plot_summary["delta_r"] = plot_summary["Transformed r"] - plot_summary["Orig r"]

    runs_present = sorted(plot_summary["Run"].unique())
    x = np.arange(len(TRANSFORM_TYPES))
    bar_w = 0.8 / len(runs_present)
    for i, run_label in enumerate(runs_present):
        sub = plot_summary[plot_summary["Run"] == run_label].set_index("Transform").reindex(TRANSFORM_TYPES)
        offsets = x - 0.4 + bar_w * i + bar_w / 2
        ax.bar(offsets, sub["delta_r"], width=bar_w, label=run_label)

    ax.axhline(0, color="k", lw=1)
    ax.set_xticks(x)
    ax.set_xticklabels(TRANSFORM_TYPES)
    ax.set_ylabel("Mean Δ Pearson r  (transformed − original)")
    ax.set_title("Accuracy shift under rigid transforms, by model and transform type")
    ax.legend(fontsize=8, loc="best")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Decision

*Fill in once the sweep runs against real Phase 1/2 checkpoints. Template:*

| Model | Features | `translation` PredPred corr | `rotation` PredPred corr | `rotation+translation` PredPred corr |
|---|---|---|---|---|
| Attention | off | ? | ? | ? |
| Attention | on  | ? | ? | ? |
| Distance  | off | ? | ? | ? |
| Distance  | on  | ? | ? | ? |

**Does invariance hold well enough to skip Tier-5 equivariant architectures?**
*If `features off` models show `PredPred corr ≈ 1.0` for rotation as
predicted, and `features on` models show only a small drop, that's the
"strong argument against the complexity cost of full equivariance" outcome
SUMMER_PLAN.md anticipated. If `features on` shows a large drop, the
practical fix is likely simpler than full equivariance: drop `normal` (per
notebook 08/10's historical precedent of finding it marginal-to-harmful
anyway) rather than investing in an equivariant architecture — worth
checking whether the accuracy lost by dropping `normal` is smaller than the
accuracy lost to rotational instability before concluding either way.*

**If instability is found:** the SUMMER_PLAN fallback — coordinate
augmentation during training (random rotations applied to the training set)
— would need its own follow-up ablation, not assumed to fix things without
verification.
